# 02 — What the power spectrum misses

We compare a non-Gaussian field with a phase-scrambled field that has exactly the same Fourier amplitudes. Then we ask which summaries can distinguish them.

In [ ]:
from pathlib import Path
import numpy as np
import sys, os
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ROOT = Path(os.path.dirname(os.path.abspath('.'))).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 2601
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "gray": "#626C78",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(0.995, 0.005, "analytic-fixture", ha="right", va="bottom",
             fontsize=7, color=COLORS["gray"])
    fig.savefig(OUTPUT_DIR / name, bbox_inches="tight")

In [ ]:
n, box_size = 192, 400.0

def make_nongaussian_field(seed):
    rng = np.random.default_rng(seed)
    k_axis = 2 * np.pi * np.fft.fftfreq(n, d=box_size / n)
    kx, ky = np.meshgrid(k_axis, k_axis, indexing="ij")
    smooth = np.exp(-0.5 * (np.hypot(kx, ky) / 0.18) ** 4)
    gaussian = np.fft.ifft2(
        np.fft.fft2(rng.normal(size=(n, n))) * smooth
    ).real
    gaussian = (gaussian - gaussian.mean()) / gaussian.std()
    rho = np.exp(0.82 * gaussian)
    return rho / rho.mean() - 1.0


field_original = make_nongaussian_field(SEED + 100)

## Hold $P(k)$ fixed

Phases from the FFT of a real random field automatically have the Hermitian symmetry needed for a real inverse transform. A common rescaling keeps both density contrasts above $-1$ without changing their equality in Fourier amplitude.

In [ ]:
def phase_scramble(field, seed):
    rng = np.random.default_rng(seed)
    amplitudes = np.abs(np.fft.fft2(field))
    amplitudes[0, 0] = 0.0
    random_field_k = np.fft.fft2(rng.normal(size=field.shape))

    # TODO 1: combine the original amplitudes with the random phases.
    raise NotImplementedError


field_scrambled = phase_scramble(field_original, SEED + 101)
deepest_trough = max(-field_original.min(), -field_scrambled.min())
common_scale = min(1.0, 0.95 / deepest_trough)
field_original *= common_scale
field_scrambled *= common_scale

amp_original = np.abs(np.fft.fft2(field_original))
amp_scrambled = np.abs(np.fft.fft2(field_scrambled))
amplitude_error = np.max(np.abs(amp_original - amp_scrambled)) / amp_original.max()

print(f"minimum delta: {field_original.min():.3f}, {field_scrambled.min():.3f}")
print(f"Fourier-amplitude difference: {amplitude_error:.2e}")
assert min(field_original.min(), field_scrambled.min()) > -1
assert amplitude_error < 1e-12

In [ ]:
def power_spectrum_2d(field, box_size, edges):
    n = field.shape[0]
    area = box_size**2
    pixel_area = (box_size / n) ** 2
    field_k = pixel_area * np.fft.fft2(field)
    power_modes = np.abs(field_k)**2 / area
    k_axis = 2 * np.pi * np.fft.fftfreq(n, d=box_size / n)
    kx, ky = np.meshgrid(k_axis, k_axis, indexing="ij")
    k_modes = np.hypot(kx, ky)
    shell = np.digitize(k_modes.ravel(), edges) - 1
    valid = (shell >= 0) & (shell < len(edges) - 1)
    counts = np.bincount(shell[valid], minlength=len(edges) - 1)
    power_sum = np.bincount(
        shell[valid], weights=power_modes.ravel()[valid], minlength=len(edges) - 1
    )
    power = np.divide(
        power_sum, counts, out=np.full(len(counts), np.nan), where=counts > 0
    )
    return 0.5 * (edges[:-1] + edges[1:]), power, counts


k_fundamental = 2 * np.pi / box_size
k_nyquist = np.pi / (box_size / n)
edges = np.arange(0.5 * k_fundamental, k_nyquist + 2.5 * k_fundamental,
                  2 * k_fundamental)
k, p_original, counts = power_spectrum_2d(field_original, box_size, edges)
_, p_scrambled, _ = power_spectrum_2d(field_scrambled, box_size, edges)
valid = (counts > 0) & (p_original > 0)
power_error = np.max(np.abs(p_scrambled[valid] / p_original[valid] - 1))
print(f"largest shell-power difference: {power_error:.2e}")
assert power_error < 1e-10

both = np.concatenate([field_original.ravel(), field_scrambled.ravel()])
lo, hi = np.quantile(both, [0.02, 0.98])
norm = TwoSlopeNorm(vmin=lo, vcenter=0.0, vmax=hi)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, field, title in zip(
    axes[:2],
    [field_original, field_scrambled],
    ["Original", "Phase-scrambled"],
):
    image = ax.imshow(field.T, origin="lower", cmap="RdBu_r", norm=norm,
                      extent=[0, box_size, 0, box_size])
    ax.set(title=title, xlabel=r"$x\;[h^{-1}\,\mathrm{Mpc}]$",
           ylabel=r"$y\;[h^{-1}\,\mathrm{Mpc}]$")
    ax.grid(False)
fig.colorbar(image, ax=axes[:2], label=r"$\delta$")
axes[2].loglog(k[valid], p_original[valid], color=COLORS["blue"], label="original")
axes[2].loglog(k[valid], p_scrambled[valid], "--", color=COLORS["orange"],
               label="phase-scrambled")
axes[2].set(title="Identical power spectra", xlabel=r"$k$",
            ylabel=r"$P_{2D}(k)$")
axes[2].legend()
savefig(fig, "02_same_power_mystery.png")
plt.show()

## Use phase-sensitive summaries

A selected bispectrum multiplies three shell-filtered maps, so its spatial mean measures a chosen closed-triangle coupling. Unlike $P(k)$, it retains information about relationships among Fourier phases.

In [ ]:
def shell_filter(field, box_size, center, width):
    n = field.shape[0]
    k_axis = 2 * np.pi * np.fft.fftfreq(n, d=box_size / n)
    kx, ky = np.meshgrid(k_axis, k_axis, indexing="ij")
    mask = np.abs(np.hypot(kx, ky) - center) < width / 2
    return np.fft.ifft2(np.fft.fft2(field) * mask).real


def bispectrum_response(field, box_size, triangle, width):
    filtered = [shell_filter(field, box_size, center, width) for center in triangle]
    # TODO 2: return <d1 d2 d3> / sqrt(<d1^2><d2^2><d3^2>).
    raise NotImplementedError

In [ ]:
width = 3 * k_fundamental
triangles = {
    "low equilateral": (0.08, 0.08, 0.08),
    "high equilateral": (0.18, 0.18, 0.18),
    "squeezed": (0.045, 0.16, 0.16),
}
bis_original = np.array([
    bispectrum_response(field_original, box_size, triangle, width)
    for triangle in triangles.values()
])
bis_scrambled = np.array([
    bispectrum_response(field_scrambled, box_size, triangle, width)
    for triangle in triangles.values()
])

print("bispectrum response, original:     ", np.round(bis_original, 3))
print("bispectrum response, phase-scrambled:", np.round(bis_scrambled, 3))
assert not np.allclose(bis_original, bis_scrambled)

bins = np.linspace(np.quantile(both, 0.002), np.quantile(both, 0.995), 55)
centers = 0.5 * (bins[:-1] + bins[1:])
pdf_original, _ = np.histogram(field_original, bins=bins, density=True)
pdf_scrambled, _ = np.histogram(field_scrambled, bins=bins, density=True)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.7), constrained_layout=True)
axes[0].semilogy(centers, pdf_original, color=COLORS["blue"], label="original")
axes[0].semilogy(centers, pdf_scrambled, "--", color=COLORS["orange"],
                 label="phase-scrambled")
axes[0].set(title="One-point PDF", xlabel=r"$\delta$", ylabel="density")
axes[0].legend()

positions = np.arange(len(triangles))
axes[1].bar(positions - 0.18, bis_original, 0.36, color=COLORS["blue"],
            label="original")
axes[1].bar(positions + 0.18, bis_scrambled, 0.36, color=COLORS["orange"],
            label="phase-scrambled")
axes[1].set(title="Selected bispectrum", ylabel="normalized response",
            xticks=positions, xticklabels=["low eq.", "high eq.", "squeezed"])
axes[1].legend()
savefig(fig, "02_summary_fingerprint.png")
plt.show()

## A small inverse-problem example

If $x=\theta^2+\epsilon$, a positive observation is compatible with two regions of parameter space. Simulating $x$ is easy even when the likelihood of a much richer cosmological summary is not.

In [ ]:
theta = np.linspace(-3, 3, 1200)
x_observed, noise_sigma = 2.25, 0.40
posterior = np.exp(-0.5 * ((x_observed - theta**2) / noise_sigma) ** 2)
posterior /= posterior.sum() * (theta[1] - theta[0])

fig, ax = plt.subplots(figsize=(7, 3.3), constrained_layout=True)
ax.plot(theta, posterior, color=COLORS["purple"], lw=2)
ax.fill_between(theta, posterior, color=COLORS["purple"], alpha=0.15)
ax.set(title="One observation, two parameter regions",
       xlabel=r"$\theta$", ylabel=r"$p(\theta\mid x_{\rm obs})$")
ax.set_yticks([])
savefig(fig, "02_sbi_teaser.png")
plt.show()

**Conclusion.** $P(k)$ preserves variance as a function of scale but discards phase organization. The one-point PDF and selected bispectrum retain different parts of that missing information; neither is automatically sufficient for every inference problem.